In [1]:
import numpy as np
import pandas as pd

In [2]:
data = pd.read_csv("creditcard.csv")

In [3]:
data.shape

(284807, 31)

In [4]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [5]:
data.Class.value_counts(normalize=True)

Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64

# <u>IMPORTANT CONSIDERATIONS<u>
- **1 : Fraud** 
- **0 : No Fraud**
- Here, we are using "No Fraud" loosely. It is more appropriate to say that "Fraud : Not Reported".
- In reality, a lot Fraud events do not get reported. Therefore, the data is bound to be **imbalanced**.
- All we can conclude is "**Fraud**" vs "**Not Reported as Fraud**"

## <U>DATA CHECK<U>
- We check for missing values
- We check for duplicate Data points

In [8]:
data.isnull().sum().head()

Time    0
V1      0
V2      0
V3      0
V4      0
dtype: int64

In [9]:
data.duplicated().sum()

1081

In [10]:
data_clean = data.drop_duplicates()

In [11]:
data_clean.shape

(283726, 31)

- There are no misisng values.
- However, there are duplicate rows. We have dropped them. It is necessary to drop them.


## <u>DATA PREPARATION and ML Algorithm : XGBoost<u>
- We will be using XGBoost to train the model.
- Splitting the data into Train and Test
- Since this is a heavily skewed data. We need to use *RandomizedSearchCV, StratifiedKFold* methods to make sure the model learns correctly.
- **<u>StratifiedKFold<u>** : It tries to preserve the same proportion of each class in every fold. This step is critical as Fraud is "rare" or "rarely reported".
- **<u>RandomizedSearchCV<u>** : XGBoost has hyperparameters that we need before training the model.RandomizedSearchCV gives the optimal choice for hyperparameters using **cross validation**.


In [14]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler

In [15]:
X = data_clean.drop(columns="Class")
Y = data_clean.Class

In [16]:
X_temp, X_test, y_temp, y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=100)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=100)

In [32]:
CrossV = StratifiedKFold(n_splits=3, shuffle=True, random_state=100)

- **CrossV** : makes sure that each search split has the same fraud rate.

## <u>XGBoost & Hyperparamter Tuning ( Search )<u>
- Step 1 : Check the imbalance between the 2 classes ( Fraud vs Genuine )
- Step 2 : Find an optimal combination of XGBoost Settings.

In [37]:
r = (y_train == 0).sum() / (y_train == 1).sum() 

In [45]:
print ("Ratio of Genuine to Fraud transactions :" , round(r,0))

Ratio of Genuine to Fraud transactions : 598.0


In [ ]:
from xgboost import XGBClassifier

In [63]:
xgb_tune = RandomizedSearchCV(
    XGBClassifier(eval_metric="aucpr", tree_method="hist", random_state=42, n_jobs=-1),
    param_distributions={
        "n_estimators":     [200, 400, 600],
        "max_depth":        [3, 4, 5, 6, 8],
        "learning_rate":    [0.03, 0.05, 0.1],
        "subsample":        [0.7, 0.85, 1.0],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "min_child_weight": [1, 3, 5],
        "scale_pos_weight": [1, np.sqrt(r), r],
    },
    n_iter=10, scoring="average_precision", cv=CrossV, random_state=100, n_jobs=1)
xgb_tune.fit(X_train, y_train)
xgb = xgb_tune.best_estimator_

## <U>FRAUD PROBABILITES<U>

In [69]:
prob_val  =  xgb.predict_proba(X_val)[:, 1]
prob_test =  xgb.predict_proba(X_test)[:, 1]

### <u>VALIDATION THRESHOLD<U> 
- We now compute the Validation threshold, using the validation set only.
- We then apply that threshold to the test set.
- We measure using the F1 Score and retrieve the threshold corresponding to the highest F1 Score.

In [109]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, average_precision_score,
                             precision_recall_curve, confusion_matrix)

In [111]:
pr,re,thr = precision_recall_curve(y_val, prob_val)

In [86]:
F1 = 2 * pr[:-1] * re[:-1] / (pr[:-1] + re[:-1] )

In [88]:
ind = np.argmax(F1)

In [90]:
opt_thr = thr[ind]

In [94]:
print("Optimum Threshold : " , opt_thr)

Optimum Threshold :  0.32015863


### <U>FEATURES<U> 

In [102]:
f =pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)

In [106]:
print(f.head(10))

V14    0.214993
V10    0.202268
V12    0.202164
V17    0.058583
V7     0.036065
V11    0.029905
V4     0.021430
V20    0.014947
V26    0.012803
V16    0.012732
dtype: float32


- We now convert probabilities into Prediction and then check how the model performed.

# <U>TEST METRICS<U>

In [163]:
prediction = (prob_test >= opt_thr ).astype(int)

In [165]:
prediction

array([0, 0, 0, ..., 0, 0, 0])

In [190]:
Accuracy = accuracy_score(y_test, prediction)
precision = precision_score(y_test, prediction)
Recall = recall_score(y_test, prediction)
RO_AC = roc_auc_score(y_test, prediction)
Conf_matr = confusion_matrix(y_test, prediction)
AUPRC = average_precision_score(y_test, prediction)

In [194]:
print("Accuracy Score: " , Accuracy)
print("Precision Score: " , precision)
print("Recall Score: " , Recall)
print("ROC_AUC_Score: " , RO_AC)
print("AUPRC Score: " , AUPRC)
print("\nConfusion Matrix:\n" , Conf_matr)


Accuracy Score:  0.9994889507630493
Precision Score:  0.9024390243902439
Recall Score:  0.7789473684210526
ROC_AUC_Score:  0.8894030764542643
AUPRC Score:  0.7033225733463448

Confusion Matrix:
 [[56643     8]
 [   21    74]]


# <U>CONCLUSION<U>

- Point to be noted. A high accuracy doesn not automatically imply that the model is good.
- A useless model will also high accuracy owning to the severe imbalance in the dataset.
- **FOCUS** : *Precision, Recall, AUPRC*
- Precision of 90.24% indicates that most transactions flagged as fraudulent are genuine fraud transactions.
- Recall of 77.89% indicates that the model detects approximately 78% of actual fraudulent transactions, although some fraud cases remain undetected.
- AUPRC score of 0.7033 indicates a meaningful ability to identify fraudulent transactions while accounting for the trade-off between precision and recall.